In [2]:
# fix_2_embeddings.py
from sentence_transformers import SentenceTransformer
import numpy as np

In [3]:
# Load two embedding models
generic_model = SentenceTransformer("all-MiniLM-L6-v2")
domain_model  = SentenceTransformer("BAAI/bge-base-en-v1.5")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Domain-specific query and semantically equivalent documents
test_cases = [
    {
        "query": "My subscription was charged twice this month",
        "relevant": "Duplicate billing issue — customer charged twice in billing cycle",
        "irrelevant": "How do I update my payment method in account settings?"
    },
    {
        "query": "The API gateway returned a 429 throttle error on Pro tier",
        "relevant": "User is being rate limited on their current plan",
        "irrelevant": "Upgrade your plan to get more API calls per minute"
    },
    {
        "query": "I cannot log into my account after the password reset",
        "relevant": "Authentication failure after password change — session token invalid",
        "irrelevant": "Our servers experienced downtime on March 12th"
    }
]

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("=" * 65)
print(f"{'Test Case':<5} | {'Model':<10} | {'Relevant Sim':>12} | {'Irrelevant Sim':>14}")
print("=" * 65)

for i, case in enumerate(test_cases):
    for model_name, model in [("Generic", generic_model), ("Domain", domain_model)]:
        q   = model.encode(case["query"])
        rel = model.encode(case["relevant"])
        irr = model.encode(case["irrelevant"])

        sim_rel = cosine_sim(q, rel)
        sim_irr = cosine_sim(q, irr)

        flag = "✅" if sim_rel > sim_irr else "❌"
        print(f"  TC {i+1}   | {model_name:<10} | {sim_rel:>12.4f} | {sim_irr:>14.4f}  {flag}")
    print("-" * 65)

Test Case | Model      | Relevant Sim | Irrelevant Sim
  TC 1   | Generic    |       0.6189 |         0.3053  ✅
  TC 1   | Domain     |       0.8519 |         0.6052  ✅
-----------------------------------------------------------------
  TC 2   | Generic    |       0.2754 |         0.4297  ❌
  TC 2   | Domain     |       0.5799 |         0.5973  ❌
-----------------------------------------------------------------
  TC 3   | Generic    |       0.5694 |         0.1306  ✅
  TC 3   | Domain     |       0.7276 |         0.5417  ✅
-----------------------------------------------------------------
